## LSTM Model for Energy Forecasting

To complement the Random Forest baseline, we implement a Long Short-Term Memory (LSTM) neural network for time-series forecasting.

LSTM models are a type of recurrent neural network designed to learn long-term dependencies in sequential data. Unlike tree-based models, LSTMs process data as ordered sequences, making them well-suited for energy forecasting tasks.

In this project, the LSTM model is used to predict:
- Photovoltaic (PV) energy generation
- Household grid import (energy consumption)

In [12]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

import matplotlib.pyplot as plt



Loading the dataset

In [13]:
df = pd.read_csv(r"C:\Users\test\OneDrive\Documents\GitHub\DataScienceCapstone\data\residential3_cleaned.csv")

df = df.copy()

# convert timestamp
df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'])
df = df.sort_values('utc_timestamp')

# time features
df['hour'] = df['utc_timestamp'].dt.hour
df['day'] = df['utc_timestamp'].dt.day
df['month'] = df['utc_timestamp'].dt.month
df['day_of_week'] = df['utc_timestamp'].dt.dayofweek

# optional: weekend flag
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Ensure it's a datetime format
df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'])

# Set as index and sort chronologically
df = df.set_index('utc_timestamp').sort_index()

# Preview
print(df.head())

                           DE_KN_residential3_circulation_pump  \
utc_timestamp                                                    
2016-02-28 17:30:00+00:00                                0.018   
2016-02-28 17:45:00+00:00                                0.018   
2016-02-28 18:00:00+00:00                                0.018   
2016-02-28 18:15:00+00:00                                0.018   
2016-02-28 18:30:00+00:00                                0.018   

                           DE_KN_residential3_dishwasher  \
utc_timestamp                                              
2016-02-28 17:30:00+00:00                          0.001   
2016-02-28 17:45:00+00:00                          0.000   
2016-02-28 18:00:00+00:00                          0.001   
2016-02-28 18:15:00+00:00                          0.001   
2016-02-28 18:30:00+00:00                          0.000   

                           DE_KN_residential3_freezer  \
utc_timestamp                                           
20

Cyclic encoding

In [14]:
import numpy as np

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['dayofweek_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dayofweek_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Now drop the old linear columns so they don't confuse the model
df = df.drop(columns=['hour', 'day', 'month', 'day_of_week'])

Feature selection

In [15]:
feature_cols = [
    'DE_KN_residential3_circulation_pump',
    'DE_KN_residential3_dishwasher',
    'DE_KN_residential3_freezer',
    'DE_KN_residential3_refrigerator',
    'DE_KN_residential3_washing_machine',

    'hour_sin', 'hour_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'month_sin', 'month_cos',
    'is_weekend'
]

target_cols = [
    'DE_KN_residential3_pv',
    'DE_KN_residential3_grid_import'
]

Scaling

In [23]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaled_X = scaler_X.fit_transform(df[feature_cols])
scaled_y = scaler_y.fit_transform(df[target_cols])


Here I am implementing a sliding window approach due to the nature of the data i.e. time series

In [24]:
def create_sequences(X, y, time_steps):
    Xs, ys = [], []

    # Loop through the data, stopping before we run out of room for a full window
    for i in range(len(X) - time_steps):
        # Grab the past 'time_steps' rows for features
        window_X = X[i : i + time_steps]

        # Grab the target at the VERY NEXT time step after the window
        target_y = y[i + time_steps]

        Xs.append(window_X)
        ys.append(target_y)

    return np.array(Xs), np.array(ys)

Building the sequences

In [25]:
TIME_STEPS = 96

X_seq, y_seq = create_sequences(scaled_X, scaled_y, TIME_STEPS)
print(f"X shape: {X_seq.shape} -> (Samples, Time Steps, Features)")
print(f"y shape: {y_seq.shape} -> (Samples, Targets)")

X shape: (47514, 96, 12) -> (Samples, Time Steps, Features)
y shape: (47514, 2) -> (Samples, Targets)


Train test split


In [26]:
split = int(len(X_seq) * 0.8)

X_train = X_seq[:split]
y_train = y_seq[:split]

X_test = X_seq[split:]
y_test = y_seq[split:]

print(f"Training data: {X_train.shape[0]} samples")
print(f"Testing data: {X_test.shape[0]} samples")

Training data: 38011 samples
Testing data: 9503 samples


Building the LSTM Model

In [27]:
from tensorflow.keras.layers import GRU
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

model = Sequential()

# Layer 1: The wide receiving layer
# return_sequences=True is REQUIRED if you are stacking another LSTM after this
model.add(LSTM(units=64,
               input_shape=(X_train.shape[1], X_train.shape[2]),
               return_sequences=True,
               kernel_regularizer=l2(0.001))) # L2 penalty reduces overfitting

# Aggressive dropout
model.add(Dropout(0.3))

# Layer 2: The compression layer
# return_sequences=False because we are moving to Dense layers next
model.add(LSTM(units=32,
               return_sequences=False,
               kernel_regularizer=l2(0.001)))

model.add(Dropout(0.3))

# Final Dense processing
model.add(Dense(16, activation='relu'))

# Output Layer (1 unit because we dropped PV)
model.add(Dense(1))
optimizer = Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='mse',metrics=['mae'])
model.summary()

C:\Users\test\OneDrive\Documents\GitHub\DataScienceCapstone\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 96, 64)         │        19,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 96, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,673 (127.63 KB)

 Trainable params: 32,673 (127.63 KB)

 Non-trainable params: 0 (0.00 B)

The training


In [29]:
early_stop = EarlyStopping(monitor='val_loss',
                           patience=5,
                           restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 70s 65ms/step - loss: 0.0174 - mae: 0.0515 - val_loss: 0.0079 - val_mae: 0.0556
Epoch 2/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 64s 60ms/step - loss: 0.0069 - mae: 0.0512 - val_loss: 0.0070 - val_mae: 0.0557
Epoch 3/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 63s 59ms/step - loss: 0.0065 - mae: 0.0508 - val_loss: 0.0072 - val_mae: 0.0529
Epoch 4/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 70s 65ms/step - loss: 0.0065 - mae: 0.0508 - val_loss: 0.0069 - val_mae: 0.0521
Epoch 5/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 81s 75ms/step - loss: 0.0064 - mae: 0.0507 - val_loss: 0.0073 - val_mae: 0.0525
Epoch 6/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 81s 75ms/step - loss: 0.0064 - mae: 0.0506 - val_loss: 0.0069 - val_mae: 0.0534
Epoch 7/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 58s 54ms/step - loss: 0.0064 - mae: 0.0506 - val_loss: 0.0070 - val_mae: 0.0537
Epoch 8/10
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 59s 55ms/step - loss: 0.0064 - mae: 0.0505 - val_loss: 0.0069 - val_mae: 0.0540
Epoch 9/10
1070/1070 ━━━

Predictions and inverse scaling

In [22]:
y_pred = model.predict(X_test)

#y_test_2d = y_test.reshape(-1, y_pred.shape[1])

y_pred_inv = scaler_y.inverse_transform(y_pred)
y_test_inv = scaler_y.inverse_transform(y_test)

297/297 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step


ValueError: non-broadcastable output operand with shape (9503,1) doesn't match the broadcast shape (9503,2)

Evaluation

In [30]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 1. Generate predictions and reverse the scaling
y_pred = model.predict(X_test)

y_pred_inv = scaler_y.inverse_transform(y_pred)
y_test_inv = scaler_y.inverse_transform(y_test)

# 2. Slice the arrays
# Since we dropped PV, Grid Import is now the ONLY column (index 0)
grid_actual = y_test_inv[:, 0]
grid_pred = y_pred_inv[:, 0]

# 3. Calculate metrics for Grid Import
grid_mae = mean_absolute_error(grid_actual, grid_pred)
grid_rmse = np.sqrt(mean_squared_error(grid_actual, grid_pred))
grid_r2 = r2_score(grid_actual, grid_pred)

# 4. Print the results
print(f"--- Grid Import Forecast ---")
print(f"MAE:  {grid_mae:.3f} kW")
print(f"RMSE: {grid_rmse:.3f} kW")
print(f"R²:   {grid_r2:.3f}")

297/297 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step


ValueError: non-broadcastable output operand with shape (9503,1) doesn't match the broadcast shape (9503,2)

Visualization to compare the actual vs predicted plots

In [ ]:
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
# 96 steps = 1 day (at 15-min intervals). Let's look at a 4-day window.
PLOT_STEPS = 96 * 4

# Slice the arrays to only plot the first few days of the test set
grid_actual_plot = grid_actual[:PLOT_STEPS]
grid_pred_plot = grid_pred[:PLOT_STEPS]

# Create a single, wide figure
plt.figure(figsize=(15, 6))

# --- PLOT: Grid Import ---
plt.plot(grid_actual_plot, label='Actual Grid Import', color='green', linewidth=2)
plt.plot(grid_pred_plot, label='Predicted Grid Import', color='red', linestyle='--', linewidth=2)

plt.title('Grid Import: Actual vs. Predicted (4-Day Window)', fontsize=14)
plt.ylabel('Power (kW)', fontsize=12)
plt.xlabel('Time Steps (15-min intervals)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()